# Zomato Play Store Review Analysis

Scraped and analyzed 2,000 most recent Indian Play Store reviews to identify user pain points, sentiment patterns, and churn signals. Built entirely from public data using Python.

**Tools used:** google-play-scraper, TextBlob, Pandas, Seaborn, WordCloud  
**Output:** Segmented dataset + visualizations + product insights report

*Note: Re-running this notebook will scrape fresh reviews. Numbers may vary slightly from the original report generated on June 26, 2026.*

In [ ]:
# Setup
!pip install google-play-scraper textblob wordcloud -q

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from google_play_scraper import reviews, Sort
from textblob import TextBlob

os.makedirs('zomato_analysis/data', exist_ok=True)
os.makedirs('zomato_analysis/visuals', exist_ok=True)
os.makedirs('zomato_analysis/report', exist_ok=True)

print("Setup complete.")

## 1. Data Collection & Cleaning
---


*   Scraping data from 2000 publicly visible user reviews on Google Play Store for the food delivery app Zomato
*   Using a python library called 'google-play-scraper' for the same, and saving the generated data as a .csv file with the details of the reviewer's ID, username, the content/comment written by them, the star-rating, which app version the rating was given for, and a blank column 'score'.
* Only the useful columns were kept while the others were dropped. A column 'review_length' was added to see how many were of a good length genuine reviews and how many were reviews contained low quality text like 'good' or 'bad app'.








In [ ]:
result, _ = reviews('com.application.zomato', lang='en', country='in', sort=Sort.NEWEST, count=2000)
df = pd.DataFrame(result)

df = df[['reviewId', 'userName', 'content', 'score', 'thumbsUpCount', 'at', 'appVersion']]
df['at'] = pd.to_datetime(df['at'])
df = df.dropna(subset=['content'])
df = df[df['content'].str.strip() != '']
df['review_length'] = df['content'].str.split().str.len()

print("Shape:", df.shape)
print("Date range:", df['at'].min(), "to", df['at'].max())
print("\nScore distribution:")
print(df['score'].value_counts().sort_index())

## 2. Sentiment Analysis
---
* Since the comments are written by the people in spoken languages, a Natural Language Processing (NLP) libray called *TextBlob* is used to scan and assign a score from -1 for most negative to +1 for most positive sentiment.  
* *Limitation: TextBlob doesn't read Hindi or Hinglish texts properly.*

In [ ]:
df['sentiment_score'] = df['content'].apply(lambda x: TextBlob(str(x)).sentiment.polarity)
df['sentiment_label'] = df['sentiment_score'].apply(
    lambda x: 'positive' if x > 0.1 else ('negative' if x < -0.1 else 'neutral')
)

print(df['sentiment_label'].value_counts())
print("\nSentiment vs Star Rating:")
print(pd.crosstab(df['sentiment_label'], df['score']))

## 3. Keyword Tagging
---
* Defining 6 common complaint themes and tagging keywords related to them, including some misspelled variations for better coverage.
* This is presence based tagging, which only tells the frequency of occurance, not the context of usage.

In [ ]:
keywords = {
    'delivery': ['delivery', 'deliver', 'delevery', 'deliverd'],
    'cancellation': ['cancel', 'cancelled', 'cancellation'],
    'refund': ['refund', 'money', 'payment', 'paid'],
    'customer_support': ['support', 'customer care', 'helpline', 'agent'],
    'pricing': ['charge', 'expensive', 'price', 'fee', 'costly'],
    'food_quality': ['cold', 'stale', 'quality', 'taste', 'bad food']
}

for category, words in keywords.items():
    df[category] = df['content'].str.lower().str.contains('|'.join(words), na=False).astype(int)

pain_points = list(keywords.keys())
print("Pain point frequency:")
print(df[pain_points].sum().sort_values(ascending=False))

print("\n% mentions from 1-star reviews:")
for col in pain_points:
    total = df[col].sum()
    one_star = df[df['score'] == 1][col].sum()
    print(f"{col}: {round((one_star/total)*100, 1)}%")

## 4. User Segmentation
---
* Segmentation of users based on the rating given by them and the number of pain points mentioned in their reviews.
* 1-star and 2+ specific complaints = churned user
* 2 or 3-star and 1+ complaint = at risk user
* 4 or 5-star = satisfied user

* *Limitation: Can't include the reviews without text in the churn analysis*

In [ ]:
def classify_user(row):
    pain_count = row[pain_points].sum()
    if row['score'] == 1 and pain_count >= 2:
        return 'churned'
    elif row['score'] <= 3 and pain_count >= 1:
        return 'at_risk'
    elif row['score'] >= 4:
        return 'satisfied'
    else:
        return 'passive'

df['user_segment'] = df.apply(classify_user, axis=1)
df['is_complaint'] = ((df['score'] == 1) & (df[pain_points].sum(axis=1) > 0)).astype(int)

print(df['user_segment'].value_counts())

print("\nChurned user pain points:")
print(df[df['user_segment'] == 'churned'][pain_points].sum().sort_values(ascending=False))

print("\nAt-risk user pain points:")
print(df[df['user_segment'] == 'at_risk'][pain_points].sum().sort_values(ascending=False))

df.to_csv('zomato_analysis/data/zomato_reviews_segmented.csv', index=False)
print("\nData saved.")

## 5. Visualizations
---
* Worldcloud: Shows the words with highest frequencies in 1-star and 5-star reviews.
* Bar Chart: The number of mentions of pain points at each of the five ratings
* Heatmap: Co-occurrence matrix; identifies which keywords occur together how many times.

In [ ]:
# Wordcloud
one_star_text = ' '.join(df[df['score'] == 1]['content'].tolist())
five_star_text = ' '.join(df[df['score'] == 5]['content'].tolist())
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
wc1 = WordCloud(width=600, height=400, background_color='black', colormap='Reds').generate(one_star_text)
wc2 = WordCloud(width=600, height=400, background_color='white', colormap='Greens').generate(five_star_text)
axes[0].imshow(wc1); axes[0].set_title('1-Star Reviews'); axes[0].axis('off')
axes[1].imshow(wc2); axes[1].set_title('5-Star Reviews'); axes[1].axis('off')
plt.tight_layout()
plt.savefig('zomato_analysis/visuals/wordcloud_comparison.png', dpi=150)
plt.show()

# Pain by rating
pain_by_rating = df.groupby('score')[pain_points].sum()
fig, ax = plt.subplots(figsize=(10, 6))
pain_by_rating.plot(kind='bar', ax=ax, colormap='Set2')
ax.set_title('Pain Point Frequency by Star Rating')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('zomato_analysis/visuals/pain_by_rating.png', dpi=150)
plt.show()

# Co-occurrence heatmap
co_matrix = pd.DataFrame(index=pain_points, columns=pain_points, dtype=float)
for p1 in pain_points:
    for p2 in pain_points:
        co_matrix.loc[p1, p2] = ((df[p1] == 1) & (df[p2] == 1)).sum()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(co_matrix, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax)
ax.set_title('Pain Point Co-occurrence Matrix')
plt.tight_layout()
plt.savefig('zomato_analysis/visuals/cooccurrence_heatmap.png', dpi=150)
plt.show()

print("All visuals saved.")

## 6. Export
---
* Exporting all the files and saving them in a zipped folder.

In [ ]:
import shutil
shutil.make_archive('zomato_analysis', 'zip', 'zomato_analysis')

for root, dirs, files in os.walk('zomato_analysis'):
    level = root.replace('zomato_analysis', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    for file in files:
        print(f'{indent}  {file}')

print("\nzomato_analysis.zip ready to download.")